# Custom Middleware

Build custom middleware by implementing hooks that run at specific points in the agent execution flow.

<img src='middleware_location.png'>


 
# Node-style hooks (Sequential Logic)
Run sequentially at specific execution points. Use for logging, validation, and state updates.

Available hooks:
- before_agent - Before agent starts (once per invocation)
- before_model - Before each model call
- after_model - After each model response
- after_agent - After agent completes (once per invocation)

Node-style hooks are executed at distinct points in the agent's overall sequential flow (the "nodes" in the execution graph). They are best for actions you need to perform before or after a step, such as logging, validation, or modifying data.


# Wrap-style hooks
Intercept execution and control when the handler is called. Use for retries, caching, and transformation.
You decide if the handler is called zero times (short-circuit), once (normal flow), or multiple times (retry logic).

Available hooks:
- wrap_model_call - Around each model call
- wrap_tool_call - Around each tool call

Wrap-style hooks execute around a call to a function or method, effectively wrapping the entire operation. They are essential for controlling how an operation runs, such as adding retries, caching, or implementing a fallback

In [ ]:
# Node hooks
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.messages import AIMessage
from langgraph.runtime import Runtime
from typing import Any

class MessageLimitMiddleware(AgentMiddleware):
    def __init__(self, max_messages: int = 50):
        super().__init__()
        self.max_messages = max_messages

    @hook_config(can_jump_to=["end"])
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if len(state["messages"]) == self.max_messages:
            return {
                "messages": [AIMessage("Conversation limit reached.")],
                "jump_to": "end"
            }
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"Model returned: {state['messages'][-1].content}")
        return None

In [ ]:
# Wrap hooks
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from typing import Callable

class RetryMiddleware(AgentMiddleware):
    def __init__(self, max_retries: int = 3):
        super().__init__()
        self.max_retries = max_retries

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        for attempt in range(self.max_retries):
            try:
                return handler(request)
            except Exception as e:
                if attempt == self.max_retries - 1:
                    raise
                print(f"Retry {attempt + 1}/{self.max_retries} after error: {e}")

# Can create by using @ decorator-based or Class based

### Decorator-based

Quick and simple for single-hook middleware. Use decorators to wrap individual functions.
Available decorators:

Node-style:
- @before_agent - Runs before agent starts (once per invocation)
- @before_model - Runs before each model call
- @after_model - Runs after each model response
- @after_agent - Runs after agent completes (once per invocation)

Wrap-style:
- @wrap_model_call - Wraps each model call with custom logic
- @wrap_tool_call - Wraps each tool call with custom logic

Convenience:
- @dynamic_prompt - Generates dynamic system prompts

In [ ]:
from langchain.agents.middleware import (
    before_model,
    wrap_model_call,
    AgentState,
    ModelRequest,
    ModelResponse,
)
from langchain.agents import create_agent
from langgraph.runtime import Runtime
from typing import Any, Callable


@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"About to call model with {len(state['messages'])} messages")
    return None

@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            if attempt == 2:
                raise
            print(f"Retry {attempt + 1}/3 after error: {e}")

agent = create_agent(
    model="gpt-4o",
    middleware=[log_before_model, retry_model],
    tools=[...],
)

When to use decorators:

- Single hook needed
- No complex configuration
- Quick prototyping

### Class-based middleware
More powerful for complex middleware with multiple hooks or configuration. Use classes when you need to define both sync and async implementations for the same hook, or when you want to combine multiple hooks in a single middleware.

In [ ]:
from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    ModelRequest,
    ModelResponse,
)
from langgraph.runtime import Runtime
from typing import Any, Callable

class LoggingMiddleware(AgentMiddleware):
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"About to call model with {len(state['messages'])} messages")
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"Model returned: {state['messages'][-1].content}")
        return None

agent = create_agent(
    model="gpt-4o",
    middleware=[LoggingMiddleware()],
    tools=[...],
)

When to use classes:
- Defining both sync and async implementations for the same hook
- Multiple hooks needed in a single middleware
- Complex configuration required (e.g., configurable thresholds, custom models)
- Reuse across projects with init-time configuration


# Custom state schema
Middleware can extend the agent’s state with custom properties. This enables middleware to:

- <b>Track state across execution</b>: Maintain counters, flags, or other values that persist throughout the agent’s execution lifecycle
- <b>Share data between hooks</b>: Pass information from before_model to after_model or between different middleware instances
- <b>Implement cross-cutting concerns</b>: Add functionality like rate limiting, usage tracking, user context, or audit logging without modifying the core agent logic
- <b>Make conditional decisions</b>: Use accumulated state to determine whether to continue execution, jump to different nodes, or modify behavior dynamically

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.agents.middleware import AgentState, AgentMiddleware
from typing_extensions import NotRequired
from typing import Any


class CustomState(AgentState):
    model_call_count: NotRequired[int]
    user_id: NotRequired[str]


class CallCounterMiddleware(AgentMiddleware[CustomState]):
    state_schema = CustomState

    def before_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        count = state.get("model_call_count", 0)
        if count > 10:
            return {"jump_to": "end"} # < This is not included in LLM prompt, just processed by Langchain itself
        return None

    def after_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        return {"model_call_count": state.get("model_call_count", 0) + 1}


agent = create_agent(
    model="gpt-4o",
    middleware=[CallCounterMiddleware()],
    tools=[],
)

# Invoke with custom state
result = agent.invoke({
    "messages": [HumanMessage("Hello")],
    "model_call_count": 0,
    "user_id": "user-123",
})

# Execution order of middleware

When using multiple middleware, understand how they execute:

In [ ]:
agent = create_agent(
    model="gpt-4o",
    middleware=[middleware1, middleware2, middleware3],
    tools=[...],
)

In [ ]:
# Before hooks run in order:
# middleware1.before_agent()
# middleware2.before_agent()
# middleware3.before_agent()
# Agent loop starts
# middleware1.before_model()
# middleware2.before_model()
# middleware3.before_model()
# Wrap hooks nest like function calls:
# middleware1.wrap_model_call() → middleware2.wrap_model_call() → middleware3.wrap_model_call() → model
# After hooks run in reverse order:
# middleware3.after_model()
# middleware2.after_model()
# middleware1.after_model()
# Agent loop ends
# middleware3.after_agent()
# middleware2.after_agent()
# middleware1.after_agent()

Key rules:
- before_* hooks: First to last
- after_* hooks: Last to first (reverse)
- wrap_* hooks: Nested (first middleware wraps all others)

# Agent jumps

To exit early from middleware, return a dictionary with jump_to:

Available jump targets:
- 'end': Jump to the end of the agent execution (or the first after_agent hook)
- 'tools': Jump to the tools node
- 'model': Jump to the model node (or the first before_model hook)

In [ ]:
from langchain.agents.middleware import AgentMiddleware, hook_config, AgentState
from langchain.messages import AIMessage
from langgraph.runtime import Runtime
from typing import Any

class BlockedContentMiddleware(AgentMiddleware):
    @hook_config(can_jump_to=["end"])
    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        last_message = state["messages"][-1]
        if "BLOCKED" in last_message.content:
            return {
                "messages": [AIMessage("I cannot respond to that request.")],
                "jump_to": "end"
            }
        return None

# Best practices

1. Keep middleware focused - each should do one thing well
2. Handle errors gracefully - don’t let middleware errors crash the agent
3. Use appropriate hook types:
    - Node-style for sequential logic (logging, validation)
    - Wrap-style for control flow (retry, fallback, caching)
4. Clearly document any custom state properties
5. Unit test middleware independently before integrating
6. Consider execution order - place critical middleware first in the list
7. Use built-in middleware when possible

## Examples
​
### Ex 1. Dynamic model selection

In [ ]:
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

complex_model = init_chat_model("gpt-4o")
simple_model = init_chat_model("gpt-4o-mini")

class DynamicModelMiddleware(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        # Use different model based on conversation length
        if len(request.messages) > 10:
            model = complex_model
        else:
            model = simple_model
        return handler(request.override(model=model))

### Ex 2. Tool call monitoring

In [ ]:
from langchain.tools.tool_node import ToolCallRequest
from langchain.agents.middleware import AgentMiddleware
from langchain.messages import ToolMessage
from langgraph.types import Command
from typing import Callable

class ToolMonitoringMiddleware(AgentMiddleware):
    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command],
    ) -> ToolMessage | Command:
        print(f"Executing tool: {request.tool_call['name']}")
        print(f"Arguments: {request.tool_call['args']}")
        try:
            result = handler(request)
            print(f"Tool completed successfully")
            return result
        except Exception as e:
            print(f"Tool failed: {e}")
            raise

### Ex 3. Dynamically selecting tools

Select relevant tools at runtime to improve performance and accuracy.

<b>Benefits</b>:
- <b>Shorter prompts</b> - Reduce complexity by exposing only relevant tools
- <b>Better accuracy</b> - Models choose correctly from fewer options
- <b>Permission control</b> - Dynamically filter tools based on user access

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from typing import Callable


class ToolSelectorMiddleware(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        """Middleware to select relevant tools based on state/context."""
        # Select a small, relevant subset of tools based on state/context
        relevant_tools = select_relevant_tools(request.state, request.runtime)
        return handler(request.override(tools=relevant_tools))

agent = create_agent(
    model="gpt-4o",
    tools=all_tools,  # All available tools need to be registered upfront
    middleware=[ToolSelectorMiddleware()],
)